In [1]:
import polars as pl

In [2]:
inicial = (
    pl.scan_parquet(r"C:\Users\diogo.durao\Documents\covid_rj.parquet")
    .select([
        "paciente_idade",
        "paciente_enumSexoBiologico",
        "paciente_endereco_nmMunicipio",
        "paciente_endereco_uf",
        "vacina_dataAplicacao",
        "vacina_fabricante_nome",
        "vacina_categoria_nome"
    ])
    .with_columns([
        pl.col("paciente_idade").cast(pl.Int64, strict=False),
        pl.col("vacina_dataAplicacao").str.to_date("%Y-%m-%d", strict=False)
    ])
    .filter(
        pl.col("paciente_idade").is_not_null() &
        pl.col("paciente_endereco_nmMunicipio").is_not_null() &
        pl.col("paciente_endereco_uf").is_not_null()
    )
)

In [3]:
display(inicial.collect().head(5))

paciente_idade,paciente_enumSexoBiologico,paciente_endereco_nmMunicipio,paciente_endereco_uf,vacina_dataAplicacao,vacina_fabricante_nome,vacina_categoria_nome
i64,str,str,str,date,str,str
60,"""F""","""SAO GONCALO""","""RJ""",2024-06-20,"""BUTANTAN""","""Faixa Etária"""
70,"""F""","""CAMPOS DOS GOYTACAZES""","""RJ""",2021-11-11,"""PFIZER""","""Faixa Etária"""
41,"""M""","""RIO DE JANEIRO""","""RJ""",2021-12-13,"""JANSSEN""","""Faixa Etária"""
26,"""M""","""BELFORD ROXO""","""RJ""",2021-10-21,"""PFIZER""","""Faixa Etária"""
24,"""F""","""CAMPOS DOS GOYTACAZES""","""RJ""",2021-09-28,"""ASTRAZENECA/FIOCRUZ""","""Faixa Etária"""


In [4]:
df_municipios = (
    inicial
    .group_by(
        "paciente_endereco_nmMunicipio",
        "paciente_endereco_uf"
    )
    .agg([
        pl.len().alias("Total_Vacinacoes"),

        pl.col("paciente_idade")
        .mean()
        .round(2)
        .alias("Media_Idade")
    ])
)

In [5]:
df_agregado = df_municipios.collect()

In [6]:
# 3. Calcular quartis

q1 = (
    df_agregado
    .select(
        pl.col("Total_Vacinacoes")
        .quantile(0.25)
    )
    .item()
)

q3 = (
    df_agregado
    .select(
        pl.col("Total_Vacinacoes")
        .quantile(0.75)
    )
    .item()
)

print(f"Municípios abaixo de {q1:.0f} vacinações são 'Baixa Vacinação'")
print(f"Municípios acima de {q3:.0f} vacinações são 'Alta Vacinação'")

Municípios abaixo de 4 vacinações são 'Baixa Vacinação'
Municípios acima de 40 vacinações são 'Alta Vacinação'


In [7]:
# 4. Criar classificação

df_export = (
    df_agregado
    .with_columns(
        pl.when(
            pl.col("Total_Vacinacoes") > q3
        )
        .then(pl.lit("Alta Vacinação"))
        
        .when(
            pl.col("Total_Vacinacoes") < q1
        )
        .then(pl.lit("Baixa Vacinação"))
        
        .otherwise(pl.lit("Média Vacinação"))
        
        .alias("Etiqueta_Classificacao")
    )
)

In [8]:
# 5. Exportar CSV

df_export.write_csv("vacinas_rj_powerbi.csv")

print("Arquivo 'vacinas_rj_powerbi.csv' gerado com sucesso!")

Arquivo 'vacinas_rj_powerbi.csv' gerado com sucesso!
